In [ ]:
# Import necessary libraries for data manipulation and modeling
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the F1 Strategy Dataset
df = pd.read_csv('f1_strategy_dataset_v4.csv')

# Display the first few rows
df.head()

In [ ]:
# Basic statistics
df.info()


In [ ]:
df.describe()

In [ ]:

# Target distribution
df['PitNextLap'].value_counts()
df['PitNextLap'].value_counts(normalize=True)



In [ ]:
# Missing values
df.isnull().sum()

In [ ]:
# Time-series aware train/test split

# Separate data by year
train_df = df[df['Year'].isin([2022, 2023])].copy()
val_df = df[df['Year'] == 2024].copy()
test_df = df[df['Year'] == 2025].copy()

print(f"Total rows: {len(df)}")
print(f"Train size: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Val size: {len(val_df)} ({len(val_df)/len(df):.2%})")
print(f"Test size: {len(test_df)} ({len(test_df)/len(df):.2%})")
print(f"\nTrain/Val/Test ratio: {len(train_df)/len(val_df):.2f} / {len(train_df)/len(test_df):.2f}")

# Verify temporal order is preserved
print(f"\nUnique sequences in train: {train_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in val: {val_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in test: {test_df.groupby(['Driver', 'Race', 'Year']).ngroups}")

In [ ]:
# Create lagged features for Position, LapTime, and degradation metrics
# Group by Driver, Race, Year to maintain temporal order within sequences

# Create lagged features
train_df['Position_lag1'] = train_df.groupby(['Driver', 'Race', 'Year'])['Position'].shift(1)
train_df['Position_lag2'] = train_df.groupby(['Driver', 'Race', 'Year'])['Position'].shift(2)
train_df['Position_lag3'] = train_df.groupby(['Driver', 'Race', 'Year'])['Position'].shift(3)

# LapTime lags
train_df['LapTime_lag1'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime (s)'].shift(1)
train_df['LapTime_lag2'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime (s)'].shift(2)
train_df['LapTime_lag3'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime (s)'].shift(3)

# Rolling average pace (using LapTime_Delta as a proxy for pace change)
# Rolling average of LapTime_Delta over 3 laps
train_df['Rolling_avg_pace'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime_Delta'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Rolling degradation (rolling average of Cumulative_Degradation)
train_df['Rolling_degradation'] = train_df.groupby(['Driver', 'Race', 'Year'])['Cumulative_Degradation'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Fill NaN values (from shifts) with 0
# Only fill numeric columns with 0, leave string/categorical columns unchanged
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
train_df[numeric_cols] = train_df[numeric_cols].fillna(0)


In [ ]:
train_df = train_df.drop(['Driver', 'Race'], axis=1)

In [ ]:
# Convert Compound column to categorical with specified order
categories = ['SOFT', 'MEDIUM', 'HARD']
dtype = pd.CategoricalDtype(categories=categories, ordered=True)
train_df['Compound'] = train_df['Compound'].astype(dtype)

# Verify the conversion
train_df['Compound'].cat.categories.tolist()
train_df['Compound'].cat.codes.value_counts()

In [ ]:
# Impute missing values
imputer = SimpleImputer(strategy='most_frequent')
train_df['Compound'] = imputer.fit_transform(train_df['Compound'].values.reshape(-1, 1)).flatten()

In [ ]:
# Encode Compound column with OrdinalEncoder
encoder = OrdinalEncoder()
train_df['Compound_encoded'] = encoder.fit_transform(train_df['Compound'].to_numpy().reshape(-1, 1)).flatten()

# Drop the original Compound column
train_df.drop('Compound', axis=1, inplace=True)

In [ ]:
X_train, y_train = train_df.drop('PitNextLap', axis=1), train_df['PitNextLap']

In [ ]:
# Train LightGBM model
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)